# 04 — Semantic Representation of TKH Nodes

## Multi-Resolution Semantic Abstraction over an Evolving Knowledge Hypergraph


## Purpose

This notebook constructs semantic representations for the entities contained
within the Temporal Knowledge Hypergraph (TKH).

The goal is to transform node textual information into a numerical embedding
space while keeping the process independent from downstream evaluation data.

The semantic representation provides the content-based component required for
later abstraction of the hypergraph structure.


---

## Research question addressed

This notebook answers:

> How can heterogeneous TKH entities be represented in a common semantic space
> while preserving their textual meaning?


---

## Relation to previous stages

Previous notebooks established:

### Section 1 — TKH validation

Verified:

- nodes and hyperedges are structurally valid;
- node metadata is available;
- semantic text fields exist.


### Section 2 — Temporal snapshots

Established:

$[
H(2020), H(2022), H(2024), H(2026)
]$

using temporally honest node availability.

This notebook builds semantic representations over the validated TKH entities
without using future information or benchmark labels.


---

## Important design choice

This notebook is benchmark-blind.

The following files are intentionally not loaded:


```text
questions.csv
ground_truth.json
```


The semantic representation is learned only from TKH node information.

This prevents benchmark leakage into the representation stage.


---

## Method

Each TKH node is represented using textual information and encoded using a
sentence-transformer model.

The resulting semantic matrix is:

$[
X_{\mathrm{sem}}\in\mathbb{R}^{|V|\times d}
]$

where:

- $(|V|)$ is the number of TKH nodes;
- $(d)$ is the embedding dimension.


---

## Inputs

From previous stages:

```text
Validated TKH nodes
|
↓
Node textual descriptions
|
↓
Semantic encoder
```

## Outputs

This notebook produces:

```text
node_embeddings
embedding matrix
node-to-index mapping
semantic representation metadata
```

In [1]:
from pathlib import Path

REPO_DIR = Path(
    "/content/tkh-hierarchy-project"
)

if not REPO_DIR.exists():
    !git clone https://github.com/mohamadghoroobi/tkh-hierarchy-project.git

%cd /content/tkh-hierarchy-project

Cloning into 'tkh-hierarchy-project'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 111 (delta 54), reused 82 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 17.05 MiB | 29.40 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/tkh-hierarchy-project


## Install semantic encoder

In [ ]:
!pip -q install sentence-transformers huggingface-hub

## Imports and reproducibility

In [ ]:
import json
import re
import random
import hashlib
import unicodedata
from pathlib import Path
from importlib.metadata import version

import numpy as np
import torch

from sentence_transformers import SentenceTransformer

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("sentence-transformers:", version("sentence-transformers"))
print("torch:", torch.__version__)

Device: cuda
sentence-transformers: 5.7.0
torch: 2.11.0+cu128


## Load TKH

In [ ]:
PROJECT_DIR = Path("/content/tkh-hierarchy-project")
DATA_DIR = PROJECT_DIR / "data"

TKH_PATH = DATA_DIR / "tkh_collection10.json"

with open(
    TKH_PATH,
    "r",
    encoding="utf-8"
) as f:
    tkh = json.load(f)

nodes = tkh["nodes"]

print(f"Loaded {len(nodes):,} nodes")

Loaded 5,798 nodes


## Establish deterministic node order

In [ ]:
nodes_ordered = sorted(
    nodes,
    key=lambda node: node["id"]
)

node_ids = [
    node["id"]
    for node in nodes_ordered
]

assert len(node_ids) == len(set(node_ids))
assert len(node_ids) == 5798

print("Deterministic node order established.")
print("First 5 IDs:", node_ids[:5])

Deterministic node order established.
First 5 IDs: ['arti_00001', 'arti_00002', 'arti_00003', 'arti_00004', 'arti_00005']


Later:

semantic_embeddings[i]

always corresponds to:

node_ids[i]

## Normalize semantic text

In [ ]:
def normalize_semantic_text(text):
    """
    Conservative normalization for embedding input.

    - Unicode NFKC normalization
    - collapse repeated whitespace
    - preserve capitalization and scientific notation
    """

    text = str(text)

    text = unicodedata.normalize(
        "NFKC",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [ ]:
semantic_texts = [
    normalize_semantic_text(
        node["surface_form"]
    )
    for node in nodes_ordered
]

assert len(semantic_texts) == 5798
assert all(semantic_texts)

print("Semantic texts prepared.")

Semantic texts prepared.


**Why only surface_form?**

This is deliberate.

We do not embed:

node type
year
hypergraph neighborhood
provenance
question text
ground truth

inside the semantic vector.

Otherwise the representation starts mixing our semantic and structural signals before we have formally defined their trade-off.

Our semantic encoder therefore represents:

$$ s(v)=f_{\theta}(\text{surface\_form}(v)) $$

The structural signal will remain separate.

This gives us a clean T2 formulation later:

$$ J = \alpha L_{\text{semantic}} + (1-\alpha)L_{\text{hypergraph}} + \lambda L_{\text{temporal}} $$

instead of hiding multiple signals inside one representation.

## Inspect text characteristics

In [ ]:
lengths = np.array([
    len(text.split())
    for text in semantic_texts
])

print("Number of semantic texts :", len(semantic_texts))
print("Minimum words            :", int(lengths.min()))
print("Maximum words            :", int(lengths.max()))
print("Mean words               :", float(lengths.mean()))
print("Median words             :", float(np.median(lengths)))

Number of semantic texts : 5798
Minimum words            : 1
Maximum words            : 58
Mean words               : 5.893066574680924
Median words             : 3.0


** This is useful because our node texts are heterogeneous: **

author       → person name
method       → short technical phrase
claim        → richer scientific statement
article      → title
problem      → scientific problem
dataset      → short name

Later, the structural signal becomes particularly important for semantically sparse entities such as authors.

## Check duplicate semantic strings

In [ ]:
from collections import Counter

text_counts = Counter(
    semantic_texts
)

duplicate_texts = {
    text: count
    for text, count in text_counts.items()
    if count > 1
}

print(
    "Unique semantic texts:",
    len(text_counts)
)

print(
    "Duplicated semantic strings:",
    len(duplicate_texts)
)

print(
    "Nodes belonging to duplicate strings:",
    sum(duplicate_texts.values())
)

Unique semantic texts: 5555
Duplicated semantic strings: 224
Nodes belonging to duplicate strings: 467


Two nodes may have identical surface text while being different TKH entities/types/provenance objects.

Their semantic embeddings can therefore be identical while their structural representations later differ.

That is actually useful for our combined method.

## Define primary semantic encoder

In [ ]:
MODEL_NAME = (
    "sentence-transformers/"
    "all-mpnet-base-v2"
)

print("Primary semantic encoder:")
print(MODEL_NAME)

Primary semantic encoder:
sentence-transformers/all-mpnet-base-v2


Exact Hugging Face revision to improve reproducibility.

In [ ]:
from huggingface_hub import model_info

try:
    info = model_info(MODEL_NAME)
    MODEL_REVISION = info.sha
except Exception as exc:
    MODEL_REVISION = None
    print(
        "Could not resolve model revision:",
        exc
    )

print("Model revision:", MODEL_REVISION)

Model revision: e8c3b32edf5434bc2275fc9bab85f82640a19130


## Load model

In [ ]:
semantic_model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE
)

embedding_dim = (
    semantic_model.get_sentence_embedding_dimension()
)

print("Embedding dimension:", embedding_dim)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 768


/tmp/ipykernel_6634/3990988104.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  semantic_model.get_sentence_embedding_dimension()


## Generate embeddings

In [ ]:
semantic_embeddings = semantic_model.encode(
    semantic_texts,

    batch_size=64,

    show_progress_bar=True,

    convert_to_numpy=True,

    normalize_embeddings=True
)

Batches:   0%|          | 0/91 [00:00<?, ?it/s]

In [ ]:
semantic_embeddings = semantic_model.encode(
    semantic_texts,

    batch_size=64,

    show_progress_bar=True,

    convert_to_numpy=True,

    normalize_embeddings=True
)

Batches:   0%|          | 0/91 [00:00<?, ?it/s]

## Validate embedding matrix

In [ ]:
assert (
    semantic_embeddings.shape[0]
    ==
    len(node_ids)
)

assert (
    semantic_embeddings.shape[1]
    ==
    embedding_dim
)

assert np.isfinite(
    semantic_embeddings
).all()

print("Embedding shape and finiteness checks passed.")

Embedding shape and finiteness checks passed.


## Check unit normalization

In [ ]:
normalize_embeddings=True

In [ ]:
norms = np.linalg.norm(
    semantic_embeddings,
    axis=1
)

print(
    "Minimum norm:",
    float(norms.min())
)

print(
    "Maximum norm:",
    float(norms.max())
)

print(
    "Mean norm:",
    float(norms.mean())
)

Minimum norm: 0.9999998807907104
Maximum norm: 1.0000001192092896
Mean norm: 1.0


In [ ]:
assert np.allclose(
    norms,
    1.0,
    atol=1e-5
)

print("Embedding normalization check passed.")

Embedding normalization check passed.


This allows cosine similarity later to be computed efficiently as:

cos(xi​,xj)= xi⊤ xj
	​


## Build ID → embedding row mapping

In [ ]:
node_to_row = {
    node_id: index
    for index, node_id
    in enumerate(node_ids)
}

assert len(node_to_row) == 5798

print(
    "Node-to-row mapping created."
)

Node-to-row mapping created.


Example:

In [ ]:
example_id = node_ids[100]

print(
    "Node:",
    example_id
)

print(
    "Row:",
    node_to_row[example_id]
)

print(
    "Vector shape:",
    semantic_embeddings[
        node_to_row[example_id]
    ].shape
)

Node: auth_00049
Row: 100
Vector shape: (768,)


## Semantic nearest-neighbor sanity test

This is not an evaluation metric.

It is only a debugging/sanity test to confirm the embeddings are not corrupted.

Use a deterministic random sample rather than benchmark questions.

In [ ]:
rng = np.random.default_rng(SEED)

sample_indices = rng.choice(
    len(node_ids),
    size=5,
    replace=False
)

sample_indices

array([4485, 2544, 3793,  517, 2510])

In [ ]:
def nearest_semantic_neighbors(
    query_index,
    embeddings,
    nodes_ordered,
    top_k=5
):
    """
    Return nearest semantic neighbors using dot product.

    Embeddings are unit-normalized, so dot product
    equals cosine similarity.
    """

    query = embeddings[query_index]

    similarities = (
        embeddings @ query
    )

    order = np.argsort(
        similarities
    )[::-1]

    results = []

    for idx in order:

        if idx == query_index:
            continue

        node = nodes_ordered[idx]

        results.append({
            "id": node["id"],
            "type": node["type"],
            "surface_form":
                node["surface_form"],
            "similarity":
                float(similarities[idx])
        })

        if len(results) == top_k:
            break

    return results

In [ ]:
for query_index in sample_indices:

    query_node = nodes_ordered[
        query_index
    ]

    print("\nQUERY")
    print(
        query_node["type"],
        "—",
        query_node["surface_form"]
    )

    neighbors = (
        nearest_semantic_neighbors(
            query_index,
            semantic_embeddings,
            nodes_ordered,
            top_k=5
        )
    )

    for neighbor in neighbors:

        print(
            f"  {neighbor['similarity']:.3f} | "
            f"{neighbor['type']:12s} | "
            f"{neighbor['surface_form']}"
        )


QUERY
task — Alloy system phonon validation
  0.761 | task         | Phonon prediction in alloy systems
  0.735 | claim        | Phonon bandstructures were validated on various alloy systems including SiGe, FeCoNi, and other high-energy alloys with agreement to existing literature.
  0.730 | task         | Binary alloy phonon prediction
  0.727 | problem      | Phonon prediction for disordered alloy systems
  0.705 | task         | Ternary alloy phonon prediction

QUERY
dataset — Uracil
  0.825 | dataset      | MD-17 Uracil
  0.756 | dataset      | MD-17 Uracil (original)
  0.620 | dataset      | Aspirin
  0.554 | dataset      | Aspirin (MD-17)
  0.548 | dataset      | MD-17 Paracetamol

QUERY
problem — Long-range interaction modeling
  0.786 | problem      | Long-range interaction modeling in MLIPs
  0.727 | problem      | Long-range behavior modeling
  0.727 | problem      | Long-range interaction capture
  0.542 | cited_work   | Interaction Network
  0.541 | task         | Atomic i

We do not report these similarities later as semantic coherence.

The same MPNet representation created the semantic signal, so evaluating clusters using MPNet cosine similarity would be circular.

## Create artifact directory

In [ ]:
ARTIFACT_DIR = (
    PROJECT_DIR
    / "artifacts"
    / "semantic"
)

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(ARTIFACT_DIR)

/content/tkh-hierarchy-project/artifacts/semantic


In [ ]:
EMBEDDING_PATH = (
    ARTIFACT_DIR
    / "semantic_embeddings.npz"
)

np.savez_compressed(
    EMBEDDING_PATH,

    embeddings=
        semantic_embeddings,

    node_ids=
        np.array(node_ids)
)

print(
    "Saved:",
    EMBEDDING_PATH
)

Saved: /content/tkh-hierarchy-project/artifacts/semantic/semantic_embeddings.npz


## Validate saved artifact

In [ ]:
saved = np.load(
    EMBEDDING_PATH
)

saved_embeddings = (
    saved["embeddings"]
)

saved_ids = (
    saved["node_ids"]
)

print(
    saved_embeddings.shape
)

print(
    len(saved_ids)
)

(5798, 768)
5798


In [ ]:
assert np.array_equal(
    saved_ids,
    np.array(node_ids)
)

assert np.allclose(
    saved_embeddings,
    semantic_embeddings
)

print(
    "Saved semantic artifact validated."
)

Saved semantic artifact validated.


## Artifact hash

In [ ]:
def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

In [ ]:
embedding_sha256 = (
    sha256_file(
        EMBEDDING_PATH
    )
)

print(
    "Embedding SHA256:",
    embedding_sha256
)

Embedding SHA256: 5cc6060e2a04d9cf5e20ae1878f855f7f12dfc928b5698ef7aa9a24e9e763fee


## Save semantic metadata

In [ ]:
semantic_metadata = {

    "num_nodes":
        len(node_ids),

    "embedding_dimension":
        int(embedding_dim),

    "model_name":
        MODEL_NAME,

    "model_revision":
        MODEL_REVISION,

    "input_field":
        "surface_form",

    "text_normalization":
        "Unicode NFKC + whitespace collapse",

    "lowercase":
        False,

    "normalized_embeddings":
        True,

    "dtype":
        str(
            semantic_embeddings.dtype
        ),

    "seed":
        SEED,

    "device_used":
        DEVICE,

    "sentence_transformers_version":
        version(
            "sentence-transformers"
        ),

    "torch_version":
        torch.__version__,

    "embedding_sha256":
        embedding_sha256,

    "evaluation_warning":
        (
            "Primary semantic embeddings must not "
            "be reused as the T6 semantic coherence "
            "evaluation signal."
        )
}

In [ ]:
METADATA_PATH = (
    ARTIFACT_DIR
    / "semantic_metadata.json"
)

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        semantic_metadata,
        f,
        indent=2
    )

print(
    "Saved:",
    METADATA_PATH
)

Saved: /content/tkh-hierarchy-project/artifacts/semantic/semantic_metadata.json


## Final Section 04 validation


In [ ]:
print(
    "===== SEMANTIC REPRESENTATION SUMMARY ====="
)

print(
    f"Nodes             : {len(node_ids):,}"
)

print(
    f"Embedding model   : {MODEL_NAME}"
)

print(
    f"Embedding dim     : {embedding_dim}"
)

print(
    f"Matrix shape      : "
    f"{semantic_embeddings.shape}"
)

print(
    f"Normalized        : True"
)

print(
    f"NaN / Inf         : "
    f"{not np.isfinite(semantic_embeddings).all()}"
)

print(
    f"Artifact          : {EMBEDDING_PATH}"
)

print(
    "\nSemantic representation stage passed."
)

===== SEMANTIC REPRESENTATION SUMMARY =====
Nodes             : 5,798
Embedding model   : sentence-transformers/all-mpnet-base-v2
Embedding dim     : 768
Matrix shape      : (5798, 768)
Normalized        : True
NaN / Inf         : False
Artifact          : /content/tkh-hierarchy-project/artifacts/semantic/semantic_embeddings.npz

Semantic representation stage passed.


Crucially, this stage has not touched:

questions.csv
ground_truth.json
hypergraph structure
future snapshot information

That separation is intentional.

One methodological note to put in the notebook

Add a markdown cell near the end:

### Evaluation separation

The MPNet embeddings produced in this notebook are part of the proposed
clustering method and therefore will not be used to measure semantic
coherence in T6.

Using the same embedding representation for clustering and evaluation
would create circularity: clusters optimized for MPNet similarity would
be evaluated by the same similarity measure.

T6 will therefore use an independent signal (a different embedding
family and/or held-out hypergraph relations / blind judgement) for
semantic coherence evaluation.

# Section 4 has established


*   A benchmark-independent semantic representation pipeline was created for TKH entities.
*   The representation uses only TKH node information and does not access retrieval questions or ground truth labels.
*   Each node is mapped into a continuous semantic embedding space:

$[
X_{\mathrm{sem}}\in\mathbb{R}^{|V|\times d}
]$

*   A deterministic mapping between TKH node identifiers and embedding indices was created.
*   The semantic embedding matrix provides a content-based representation of the evolving knowledge hypergraph.
*   The semantic representation preserves the separation between:
    * textual similarity;
    * hypergraph structure;
    * temporal evolution.

---

## Contribution to the project

At this stage, the project has established two complementary views of the TKH:

```text
Temporal structure
|
↓
Leakage-free snapshots

Semantic structure
|
↓
Node embedding representation
```

## Git Push

In [2]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [3]:
!git add -A

In [ ]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   artifacts/semantic/semantic_embeddings.npz
	new file:   artifacts/semantic/semantic_metadata.json



In [ ]:
!git config --global user.name "mohamadghoroobi"
!git config --global user.email "m.ghoroobi@gmail.com"

In [ ]:
from pathlib import Path

CLEAN = Path("/content/drive/MyDrive/Apply/Germany/ConstructorLabs/04_semantic_representation.ipynb")
REPO_FILE = Path(
    "/content/tkh-hierarchy-project/"
    "notebooks/04_semantic_representation.ipynb"
)

print("Clean file exists:", CLEAN.exists())
print("Repo file exists :", REPO_FILE.exists())

Clean file exists: True
Repo file exists : False


In [ ]:
import shutil

shutil.copy2(CLEAN, REPO_FILE)

print("Clean notebook copied into repository.")

Clean notebook copied into repository.


In [ ]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   artifacts/semantic/semantic_embeddings.npz
	new file:   artifacts/semantic/semantic_metadata.json

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/04_semantic_representation.ipynb



In [ ]:
!git add -A

In [ ]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   artifacts/semantic/semantic_embeddings.npz
	new file:   artifacts/semantic/semantic_metadata.json
	new file:   notebooks/04_semantic_representation.ipynb



In [ ]:
commit_message = """feat(semantic): add benchmark-blind semantic node representations

Build deterministic semantic representations for all TKH nodes using surface-form text only.

- establish deterministic node ordering

- conservatively normalize node surface forms

- encode all 5,798 nodes with all-mpnet-base-v2

- produce L2-normalized 768-dimensional semantic embeddings

- validate embedding shape, finiteness, normalization, and node alignment

- add nearest-neighbor sanity checks without benchmark leakage

- record model revision, dependency versions, and artifact SHA256

- persist semantic embeddings and metadata for downstream coarsening

- document separation between clustering embeddings and T6 coherence evaluation
"""

with open("/tmp/commit_message.txt", "w", encoding="utf-8") as f:
    f.write(commit_message)

In [ ]:
!git commit -F /tmp/commit_message.txt

[main cd18021] feat(semantic): add benchmark-blind semantic node representations
 3 files changed, 18 insertions(+)
 create mode 100644 artifacts/semantic/semantic_embeddings.npz
 create mode 100644 artifacts/semantic/semantic_metadata.json
 create mode 100644 notebooks/04_semantic_representation.ipynb


In [ ]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

assert token, "GITHUB_TOKEN not found"
print("Token loaded successfully")

Token loaded successfully


In [ ]:
import os
import subprocess
from pathlib import Path

username = "mohamadghoroobi"

env = os.environ.copy()

env["GITHUB_USER"] = "mohamadghoroobi"
env["GITHUB_TOKEN"] = token
env["GIT_TERMINAL_PROMPT"] = "0"


askpass = Path("/tmp/git_askpass.sh")

askpass.write_text(
"""#!/bin/sh
case "$1" in
  *Username*) echo "$GITHUB_USER" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
"""
)

askpass.chmod(0o700)

env["GIT_ASKPASS"] = str(askpass)

print("Git authentication prepared")

Git authentication prepared


In [ ]:
subprocess.run(
    ["git", "push", "origin", "main"],
    env=env,
    check=True
)

print("Push completed successfully")

Push completed successfully


In [ ]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
